# QuizMark — Train model phân loại ô tô (trống / tô / tẩy)

Chạy **lần lượt từng ô từ trên xuống** (bấm ▶ hoặc `Shift + Enter`).

**Trước khi chạy lần đầu:**
1. **Runtime → Change runtime type**: chọn GPU nếu muốn (không bắt buộc — model rất nhỏ, CPU cũng chỉ vài phút).
2. Repo GitHub để **private** → bấm biểu tượng 🔑 **Secrets** ở thanh bên trái → **Add new secret**:
   tên `GITHUB_TOKEN`, giá trị là token GitHub chỉ đọc → bật **Notebook access**.
3. Các file `bubble_dataset_*.zip` xuất từ app phải nằm trong `My Drive/quizmark-ml/data/`.
   Chưa có dữ liệu thật? Đặt `USE_SYNTHETIC = True` để chạy thử toàn bộ bằng dữ liệu giả.

**Vòng lặp làm việc:** 2. Nạp dữ liệu → 3. Sửa nhãn → 4. Train → 5. Xem mẫu đáng ngờ → sửa → train lại (1–2 vòng) → 6. Export → 7. Lưu.

In [ ]:
# ===== CẤU HÌNH — sửa nếu cần =====
GITHUB_REPO   = "TEN_TAI_KHOAN/quizmark-ml"     # đổi thành <tài khoản GitHub>/quizmark-ml
DRIVE_ROOT    = "/content/drive/MyDrive/quizmark-ml"
MODEL_VERSION = 1        # tăng lên khi muốn phát hành một model mới cho app
USE_SYNTHETIC = False    # True = chạy thử bằng dữ liệu giả (chưa có dữ liệu thật)
EPOCHS        = 30

## 1. Cài đặt
Kết nối Google Drive (sẽ hỏi quyền), tải code mới nhất từ GitHub, cài thư viện.

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if "TEN_TAI_KHOAN" in GITHUB_REPO:
    raise ValueError("Hãy sửa GITHUB_REPO ở ô CẤU HÌNH phía trên thành <tài khoản GitHub>/quizmark-ml")

REPO_DIR = "/content/quizmark-ml"
shutil.rmtree(REPO_DIR, ignore_errors=True)  # luôn lấy code mới nhất
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # repo public thì không cần token
url = f"https://{token + '@' if token else ''}github.com/{GITHUB_REPO}.git"
if subprocess.run(["git", "clone", "-q", url, REPO_DIR], capture_output=True).returncode != 0:
    raise RuntimeError("Không clone được repo. Kiểm tra GITHUB_REPO, và GITHUB_TOKEN nếu repo private.")
subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://github.com/{GITHUB_REPO}.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"], check=True)

sys.path.insert(0, REPO_DIR)
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]  # chạy lại ô này sẽ nạp code mới

import torch
print("PyTorch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có (dùng CPU — vẫn đủ nhanh)")

## 2. Nạp dữ liệu
Copy các file zip từ Drive sang máy ảo Colab (đọc hàng nghìn ảnh nhỏ trực tiếp từ Drive rất chậm) rồi giải nén.

In [ ]:
import glob
from pathlib import Path
from src import data, review, synthetic

DATA_DIR, ZIP_DIR = Path("/content/data"), Path("/content/zips")
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.rmtree(ZIP_DIR, ignore_errors=True)
ZIP_DIR.mkdir(parents=True)

if USE_SYNTHETIC:
    truth = synthetic.make_synthetic_zip(ZIP_DIR, n_sheets=20)
    OVERRIDES_CSV = Path("/content/synthetic_labels_override.csv")  # không đụng vào nhãn thật trên Drive
    OVERRIDES_CSV.unlink(missing_ok=True)
    print("⚠️ ĐANG DÙNG DỮ LIỆU GIẢ")
else:
    OVERRIDES_CSV = Path(DRIVE_ROOT) / "labels_override.csv"
    zips = glob.glob(f"{DRIVE_ROOT}/data/bubble_dataset_*.zip")
    if not zips:
        raise FileNotFoundError(f"Chưa có file zip nào trong {DRIVE_ROOT}/data — xuất dữ liệu từ app trước, "
                                "hoặc đặt USE_SYNTHETIC = True để chạy thử.")
    for z in zips:
        shutil.copy(z, ZIP_DIR)

n_zips = data.extract_zips(ZIP_DIR, DATA_DIR)
overrides = data.LabelOverrides(OVERRIDES_CSV)
rv = review.Reviewer(data.load_samples(DATA_DIR, overrides, include_ignored=True), overrides)
print(f"Đã giải nén {n_zips} file zip. Nhãn sửa tay lưu tại: {OVERRIDES_CSV}\n")
print(data.summarize(data.load_samples(DATA_DIR, overrides)))

## 3. Kiểm tra & sửa nhãn

Nhãn ban đầu lấy theo ngưỡng 50% cũ của app nên có chỗ sai. Việc của bạn:
- **Ô đã tẩy / gạch chéo** đang nằm trong `filled` hoặc `empty` → đổi thành `erased`.
- **Ô tô nhạt** bị nằm trong `empty` → đổi thành `filled`.
- **Lần quét bị lỗi** (nhận sai góc phiếu, ảnh ô toàn chữ in) → bỏ cả lần quét.

Mỗi ảnh có **số thứ tự** ở trên. Sửa bằng các lệnh ví dụ bên dưới (bỏ dấu `#` rồi chạy).
Mọi bản sửa được lưu ngay vào `labels_override.csv` trên Drive — Colab ngắt kết nối cũng không mất.

Không cần xem hết hàng nghìn ô `empty`: mục 5 sẽ tự tìm các ô đáng ngờ giúp bạn.

In [ ]:
# Chỉ dùng khi đang chạy thử bằng dữ liệu giả: giả lập việc đã sửa nhãn xong
if USE_SYNTHETIC:
    synthetic.apply_truth(overrides, truth)
    print(data.summarize(data.load_samples(DATA_DIR, overrides)))

In [ ]:
rv.list_sheets()   # mỗi dòng là một lần quét; 'tỉ lệ tô' bất thường → có thể quét lỗi, xem bằng rv.show_sheet("<sheetId>")

In [ ]:
rv.show("filled", page=0)   # các ô đang có nhãn 'tô' — tìm ô đã tẩy/gạch bị tính nhầm

In [ ]:
# rv.fix("filled", [3, 17, 42], to="erased")   # ô đã tẩy/gạch bị tính là tô
# rv.fix("filled", [8], to="empty")            # ô trống bị tính là tô
# rv.show_sheet("261108-142233")               # xem các ô 'tô' của một lần quét
# rv.ignore_sheet("261108-142233")             # bỏ cả một lần quét bị lỗi (gõ phần đầu của sheetId)
# rv.show("filled", page=1)                    # trang tiếp theo

In [ ]:
rv.show("empty", page=0)    # các ô 'trống' — tìm ô tô nhạt bị bỏ sót (không cần xem hết)
# rv.fix("empty", [5], to="filled")
# rv.fix("empty", [9], to="erased")

## 4. Train
Chia dữ liệu theo **lần quét** (train / val / test), train rồi đánh giá trên tập test — kèm so sánh với ngưỡng 50% cũ.

In [ ]:
from src import train as T

samples = data.load_samples(DATA_DIR, overrides)
train_s, val_s, test_s = data.split_by_sheet(samples)
model = T.train(train_s, val_s, epochs=EPOCHS)
test_report = T.evaluate(test_s, T.predict(model, test_s))
print()
print(T.format_report(test_report))

## 5. Mẫu đáng ngờ

Các ô mà model đoán **khác** nhãn hiện tại (model càng chắc chắn thì xếp càng trước).
Chú thích dưới mỗi ảnh: `số thứ tự: nhãn hiện tại→model đoán độ tin cậy`.
Thường là **nhãn sai** → chấp nhận đề xuất của model. Sửa xong thì **chạy lại mục 4 rồi mục 5** (1–2 vòng là đủ).

In [ ]:
rv = review.Reviewer(data.load_samples(DATA_DIR, overrides, include_ignored=True), overrides)
rv.set_predictions(samples, T.predict(model, samples))
rv.show_suspicious(page=0)

In [ ]:
# rv.accept_model([0, 1, 2, 5])        # đồng ý với model cho các ô này
# rv.fix_suspicious([3], to="filled")  # hoặc tự chọn nhãn đúng: empty / filled / erased / ignore
# rv.show_suspicious(page=1)

## 6. Export ONNX + kiểm tra bằng OpenCV 4.5.3
App Android dùng OpenCV 4.5.3 nên phải kiểm tra bằng đúng phiên bản đó (chạy trong môi trường Python 3.9 riêng, lần đầu mất khoảng 1 phút).

In [ ]:
from src import export_onnx as E

OUT = Path("/content/out")
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir()
ONNX_PATH = OUT / f"bubble_classifier_v{MODEL_VERSION}.onnx"
export_info = E.export_onnx(model, ONNX_PATH)
print(export_info)
E.save_test_vectors(model, test_s, OUT / "testvectors.npz")
E.quick_check_with_current_opencv(ONNX_PATH, OUT / "testvectors.npz")

In [ ]:
!pip -q install uv
!rm -rf /content/cv453 && uv venv /content/cv453 --python 3.9 -q
!uv pip install -q --python /content/cv453/bin/python "opencv-python-headless==4.5.3.56" "numpy<2"
!/content/cv453/bin/python {REPO_DIR}/src/verify_onnx.py --model {ONNX_PATH} --vectors {OUT}/testvectors.npz

## 7. Lưu model lên Google Drive
Chỉ làm khi ô trên in ra **OK — model dùng được trong app Android**.

In [ ]:
counts = {"train": len(train_s), "val": len(val_s), "test": len(test_s)}
E.write_model_card(OUT / "model_card.json", MODEL_VERSION, test_report, counts, export_info, repo_dir=REPO_DIR)
T.save_checkpoint(model, OUT / f"bubble_classifier_v{MODEL_VERSION}.pt")

if USE_SYNTHETIC:
    print("Đang dùng dữ liệu giả → KHÔNG lưu lên Drive. Đổi USE_SYNTHETIC = False khi có dữ liệu thật.")
else:
    dest = Path(DRIVE_ROOT) / "models"
    dest.mkdir(parents=True, exist_ok=True)
    for f in [ONNX_PATH, OUT / "model_card.json", OUT / f"bubble_classifier_v{MODEL_VERSION}.pt"]:
        shutil.copy(f, dest / f.name)
    print("Đã lưu vào Google Drive:", dest)
    print(f"→ Tải {ONNX_PATH.name} và model_card.json về máy, rồi gửi đường dẫn cho Claude để tích hợp vào app.")